In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
filepath = "../Raw/WVS_Time_Series_1981-2022_csv_v5_0.csv"
df = pd.read_csv(filepath, low_memory=False)
print(f"Full dataset loaded: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

Full dataset loaded: (443488, 1046)
Memory usage: 3.81 GB


In [3]:
# Explore data structure
print("Countries:", df['COUNTRY_ALPHA'].unique()[:20])
print("Years (S003):", sorted(df['S003'].dropna().unique()))
print("\nData info:")
print(f"  Total rows: {len(df)}")
print(f"  Unique countries: {df['COUNTRY_ALPHA'].nunique()}")
print(f"  Missing values per column (%):")
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print(f"    Mean: {missing_pct.mean():.1f}%")
print(f"    Median: {missing_pct.median():.1f}%")

Countries: ['ALB' 'AND' 'ARG' 'ARM' 'AUS' 'AZE' 'BFA' 'BGD' 'BGR' 'BIH' 'BLR' 'BOL'
 'BRA' 'CAN' 'CHE' 'CHL' 'CHN' 'COL' 'CYP' 'CZE']
Years (S003): [8, 12, 20, 31, 32, 36, 50, 51, 68, 70, 76, 100, 104, 112, 124, 152, 156, 158, 170, 191, 196, 203, 214, 218, 222, 231, 233, 246, 250, 268, 275, 276, 288, 300, 320, 332, 344, 348, 356, 360, 364, 368, 376, 380, 392, 398, 400, 404, 410, 414, 417, 422, 428, 434, 440, 446, 458, 462, 466, 484, 496, 498, 499, 504, 528, 554, 558, 566, 578, 586, 604, 608, 616, 630, 634, 642, 643, 646, 682, 688, 702, 703, 704, 705, 710, 716, 724, 752, 756, 762, 764, 780, 788, 792, 800, 804, 807, 818, 826, 834, 840, 854, 858, 860, 862, 887, 894, 909]

Data info:
  Total rows: 443488
  Unique countries: 108
  Missing values per column (%):
    Mean: 0.3%
    Median: 0.0%


In [4]:
# Step 1: Select numeric value columns (survey responses)
# Exclude metadata and identifier columns
metadata_cols = ['version', 'doi', 'COUNTRY_ALPHA', 'COW_NUM', 'COW_ALPHA', 'MODE', 'S003', 'S001', 'S002VS']
value_cols = [c for c in df.columns if c not in metadata_cols and c[0].isalpha() and len(c) > 1]

# Only keep numeric columns (filter out object/string columns)
value_cols = [c for c in value_cols if pd.api.types.is_numeric_dtype(df[c])]

print(f"Selected {len(value_cols)} numeric value columns for aggregation")

# Step 2: Aggregate to country-year level
# Calculate mean of numeric responses for each country-year combination
agg_data = df.groupby(['COUNTRY_ALPHA', 'S003'])[value_cols].mean()
agg_data = agg_data.reset_index()
agg_data = agg_data.rename(columns={'COUNTRY_ALPHA': 'country', 'S003': 'year_code'})

print(f"\nAggregated data shape: {agg_data.shape}")
print(f"Countries: {agg_data['country'].nunique()}")
print(f"Year codes: {agg_data['year_code'].nunique()}")
print(f"\nFirst few rows:")
print(agg_data.iloc[:3, :6])

Selected 1037 numeric value columns for aggregation

Aggregated data shape: (108, 1039)
Countries: 108
Year codes: 108

First few rows:
  country  year_code      S004          S006          S007       S008
0     ALB          8 -1.501251  5.002501e+02  8.037053e+07  -4.000000
1     AND         20 -4.000000  1.004050e+07  1.105757e+08  -4.500249
2     ARG         32 -3.271044  4.346731e+06  2.813898e+08  61.483178


In [5]:
# S002VS is the wave (1-7, representing different survey years)
# Let's re-aggregate using wave instead of S003
print("Re-aggregating by wave (S002VS)...")
agg_data_wave = df.groupby(['COUNTRY_ALPHA', 'S002VS'])[value_cols].mean()
agg_data_wave = agg_data_wave.reset_index()
agg_data_wave = agg_data_wave.rename(columns={'COUNTRY_ALPHA': 'country', 'S002VS': 'wave'})

print(f"Aggregated data shape: {agg_data_wave.shape}")
print(f"Countries: {agg_data_wave['country'].nunique()}")
print(f"Waves: {agg_data_wave['wave'].nunique()}")
print(f"\nCountries per wave:")
print(agg_data_wave.groupby('wave').size())

Re-aggregating by wave (S002VS)...
Aggregated data shape: (306, 1039)
Countries: 108
Waves: 7

Countries per wave:
wave
1     8
2    18
3    55
4    41
5    58
6    60
7    66
dtype: int64


In [6]:
# Calculate normalized cultural distances
from sklearn.preprocessing import StandardScaler

# Normalize each column to have mean=0, std=1
scaler = StandardScaler()
agg_normalized = agg_data_wave.copy()
agg_normalized[value_cols] = scaler.fit_transform(agg_data_wave[value_cols])

# Calculate pairwise distances for each wave
def calculate_normalized_distances(wave):
    """Calculate normalized cultural distances for a wave"""
    wave_data = agg_normalized[agg_normalized['wave'] == wave]
    if len(wave_data) <= 1:
        return pd.DataFrame()
    
    countries = wave_data['country'].values
    values = wave_data[value_cols].values
    values = np.nan_to_num(values, nan=0)
    
    from scipy.spatial.distance import pdist, squareform
    distances = squareform(pdist(values, metric='euclidean'))
    
    result_list = []
    for i, country_i in enumerate(countries):
        for j, country_j in enumerate(countries):
            if i < j:  # Upper triangle only
                result_list.append({
                    'country_i': country_i,
                    'country_j': country_j,
                    'wave': wave,
                    'cultural_distance': distances[i, j]
                })
    
    return pd.DataFrame(result_list)

# Calculate for all waves
cultural_dfs = []
for wave in sorted(agg_normalized['wave'].unique()):
    wave_cultural = calculate_normalized_distances(wave)
    if len(wave_cultural) > 0:
        cultural_dfs.append(wave_cultural)

cultural_distance_df = pd.concat(cultural_dfs, ignore_index=True)

# Add year mapping
wave_to_year = {1: 1981, 2: 1990, 3: 1995, 4: 1999, 5: 2005, 6: 2010, 7: 2017}
cultural_distance_df['year'] = cultural_distance_df['wave'].map(wave_to_year)

# Final output format
cultural_distance_df = cultural_distance_df[['country_i', 'country_j', 'year', 'cultural_distance']]

# Save
output_path = '../Clean/WVS_Cultural_Distance.csv'
cultural_distance_df.to_csv(output_path, index=False)

print(f"✓ Cultural distance matrix saved")
print(f"\nShape: {cultural_distance_df.shape}")
print(f"Columns: {list(cultural_distance_df.columns)}")
print(f"Years: {sorted(cultural_distance_df['year'].unique())}")
print(f"\nSample (first 10 rows):")
print(cultural_distance_df.head(10))
print(f"\nDistance statistics:")
print(cultural_distance_df['cultural_distance'].describe())

✓ Cultural distance matrix saved

Shape: (8054, 4)
Columns: ['country_i', 'country_j', 'year', 'cultural_distance']
Years: [1981, 1990, 1995, 1999, 2005, 2010, 2017]

Sample (first 10 rows):
  country_i country_j  year  cultural_distance
0       ARG       AUS  1981          20.635321
1       ARG       FIN  1981          38.508153
2       ARG       HUN  1981          24.175829
3       ARG       JPN  1981          12.194238
4       ARG       KOR  1981          28.431140
5       ARG       MEX  1981          15.361921
6       ARG       ZAF  1981          15.624419
7       AUS       FIN  1981          42.441810
8       AUS       HUN  1981          28.690831
9       AUS       JPN  1981          19.465163

Distance statistics:
count    8054.000000
mean       24.858894
std        13.160365
min         5.349561
25%        16.481232
50%        21.618706
75%        28.506133
max       113.703656
Name: cultural_distance, dtype: float64


In [8]:
# min and max values, get row
min_row = cultural_distance_df.loc[cultural_distance_df['cultural_distance'].idxmin()]
max_row = cultural_distance_df.loc[cultural_distance_df['cultural_distance'].idxmax()]

print(min_row)
print(max_row)

country_i                 COL
country_j                 ECU
year                     2017
cultural_distance    5.349561
Name: 6517, dtype: object
country_i                   DZA
country_j                   IRN
year                       1999
cultural_distance    113.703656
Name: 1929, dtype: object


In [7]:
# Create bidirectional pairs for clean merging
# Each pair appears twice: (A,B) and (B,A)

# Create reversed pairs
reversed_df = cultural_distance_df.copy()
reversed_df.columns = ['country_j', 'country_i', 'year', 'cultural_distance']
reversed_df = reversed_df[['country_i', 'country_j', 'year', 'cultural_distance']]

# Combine original and reversed
cultural_distance_bidirectional = pd.concat([cultural_distance_df, reversed_df], ignore_index=True)

# Sort for clean output
cultural_distance_bidirectional = cultural_distance_bidirectional.sort_values(
    by=['year', 'country_i', 'country_j']
).reset_index(drop=True)

# Save
output_path_final = '../Clean/WVS_Cultural_Distance.csv'
cultural_distance_bidirectional.to_csv(output_path_final, index=False)